In [389]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision.transforms import v2
from torchinfo import summary
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report
from sklearn.preprocessing import label_binarize
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from PIL import Image
from scipy import signal
import pandas as pd
import numpy as np
import warnings
import math
import os
warnings.filterwarnings('ignore')

In [390]:
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 150)

In [391]:
while not os.path.isdir(os.path.join(os.getcwd(), 'data')):
    os.chdir("../") # set cwd to root dir

In [392]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [393]:
if device.type == 'cuda':
    print(torch.cuda.get_device_name(0))

NVIDIA GeForce RTX 3050 Ti Laptop GPU


In [394]:
writer = SummaryWriter()

In [ ]:
class ButterworthFilter(object):
    def __init__(self, cutoff, order, fs):
        self.cutoff = cutoff
        self.order = order
        self.fs = fs

    def __call__(self, input):
        if torch.is_tensor(input):
            input = input.numpy() # convert to numpy array before using signal package

        nyquist = 0.5 * self.fs
        normal_cutoff = self.cutoff / nyquist
        b, a = signal.butter(self.order, normal_cutoff, btype='low', analog=False)
        smoothed_imu_signal = signal.filtfilt(b, a, input)
        
        return torch.from_numpy(smoothed_imu_signal.copy()) # convert back to tensor

In [ ]:
class PadToLength(object):
    def __init__(self, padlen):
        self.padlen = padlen

    def __call__(self, input):
        pad = nn.ZeroPad1d((0, self.padlen - input.shape[0]))(input)
        return pad

In [ ]:
transform = v2.Compose([
    ButterworthFilter(cutoff=10.0, order=3, fs=128.0),
    PadToLength(padlen=150),
])

In [397]:
class DUO_GAIT(Dataset):
    def __init__(self, allowed_sensors, participant_id, remove_outliers=True, transform=None, target_transform=None):
        self.participant_id = participant_id
        self.allowed_sensors = allowed_sensors

        lf_control_df = pd.read_csv(f"data/DUO-GAIT/processed/OG_st_control/sub_{participant_id:02}/left_foot_core_params.csv")
        lf_control_df.rename({ "timestamps": "start_times" }, axis=1, inplace=True)
        lf_control_df['is_fatigue'] = 0
        lf_control_df['is_right_foot'] = 0
        lf_control_df['Participant'] = participant_id

        self.create_start_end_samples_strides(lf_control_df, is_control=1)

        rf_control_df = pd.read_csv(f"data/DUO-GAIT/processed/OG_st_control/sub_{participant_id:02}/right_foot_core_params.csv")
        rf_control_df.rename({ "timestamps": "start_times" }, axis=1, inplace=True)
        rf_control_df['is_fatigue'] = 0
        rf_control_df['is_right_foot'] = 1
        rf_control_df['Participant'] = participant_id

        self.create_start_end_samples_strides(rf_control_df, is_control=1)

        lf_fatigue_df = pd.read_csv(f"data/DUO-GAIT/processed/OG_st_fatigue/sub_{participant_id:02}/left_foot_core_params.csv")
        lf_fatigue_df.rename({ "timestamps": "start_times" }, axis=1, inplace=True)
        lf_fatigue_df['is_fatigue'] = 1
        lf_fatigue_df['is_right_foot'] = 0
        lf_fatigue_df['Participant'] = participant_id

        self.create_start_end_samples_strides(lf_fatigue_df, is_control=0)

        rf_fatigue_df = pd.read_csv(f"data/DUO-GAIT/processed/OG_st_fatigue/sub_{participant_id:02}/right_foot_core_params.csv")
        rf_fatigue_df.rename({ "timestamps": "start_times" }, axis=1, inplace=True)
        rf_fatigue_df['is_fatigue'] = 1
        rf_fatigue_df['is_right_foot'] = 1
        rf_fatigue_df['Participant'] = participant_id

        self.create_start_end_samples_strides(rf_fatigue_df, is_control=0)

        self.foot_strides_df = pd.concat([lf_control_df, rf_control_df, lf_fatigue_df, rf_fatigue_df], axis=0)

        if remove_outliers:
            self.foot_strides_df = self.foot_strides_df[self.foot_strides_df['is_outlier']==False].reset_index(drop=True)

        self.transform = transform
        self.target_transform = target_transform

    def create_start_end_samples_strides(self, df, is_control):
        df.sort_values(by='stride_index', inplace=True)
        df['start_samples'] = df['ic_samples'].shift(1)
        target_time = df.loc[0, 'start_times']
        
        protocol = "control" if is_control else "fatigue"

        fatigue_df = pd.read_csv(f"data/DUO-GAIT/interim/OG_st_{protocol}/sub_{self.participant_id:02}/LF.csv")
        fatigue_df.rename({ "timestamp": "Time (secs)", "Unnamed: 0": "Sample" }, axis=1, inplace=True)
        fatigue_df['Delta (secs)'] = fatigue_df['Time (secs)'] - fatigue_df['Time (secs)'].min() # delta time

        ts_eq_check = fatigue_df['Delta (secs)'].apply(lambda x: math.isclose(x, target_time, rel_tol=1e-5))
        start_sample = fatigue_df[ts_eq_check]['Sample'].item() - fatigue_df['Sample'].min()

        df.loc[0, 'start_samples'] = start_sample
        df['start_samples'] = df['start_samples'] + fatigue_df['Sample'].min()
        df['end_samples'] = df['ic_samples'] + fatigue_df['Sample'].min() - 1 # make it inclusive for ending samples too

        df['start_samples'] = df['start_samples'].astype(np.int64)
        df['end_samples'] = df['end_samples'].astype(np.int64)

    def __len__(self):
        return len(self.foot_strides_df)

    def __getitem__(self, idx):
        row = self.foot_strides_df.iloc[idx]

        imu_signals_df = pd.DataFrame()

        for sensor_location in self.allowed_sensors:
            protocol = "fatigue" if row['is_fatigue'] else "control"
            sensor_df = pd.read_csv(f"data/DUO-GAIT/interim/OG_st_{protocol}/sub_{self.participant_id:02}/{sensor_location}.csv")
            sensor_df.rename({ "timestamp": "Time (secs)", "Unnamed: 0": "Sample" }, axis=1, inplace=True)
            sensor_df = sensor_df[(sensor_df['Sample'] >= row['start_samples']) & (sensor_df['Sample'] <= row['end_samples'])].reset_index(drop=True)

            for sensor_type in ["Gyr", "Acc"]:
                for direction in ["X", "Y", "Z"]:
                    col_name = f"{sensor_type}{direction}"
                    new_col_name = f"{sensor_location}_{col_name}"

                    imu_signals_df[new_col_name] = sensor_df[col_name]

        imu_signals_np = imu_signals_df.to_numpy().transpose()
        label = row["is_fatigue"]
        
        if self.transform:
            imu_signals_np = self.transform(imu_signals_np)

        if self.target_transform:
            label = self.target_transform(label)

        return imu_signals_np, label

d = DUO_GAIT(allowed_sensors=["LL", "RL"], participant_id=2, transform=transform)

In [398]:
d[0][0].shape

torch.Size([12, 142])

In [399]:
mx = 0
mi = 0

for pid in range(1,18):
    if pid == 4 or pid == 16:
        continue
    
    dataset = DUO_GAIT(allowed_sensors=["LL"], participant_id=pid, transform=transform)
    for sample in range(0, len(dataset)):
        mi = min(mi, dataset[sample][0].shape[1])
        mx = max(mx, dataset[sample][0].shape[1])
mi, mx

ValueError: The length of the input vector x must be greater than padlen, which is 12.

In [ ]:
isic_dataset = DUO_GAIT(transform=transform)
train_images_size = len(isic_dataset)

In [ ]:
def butter_lowpass_filter(series, cutoff=10.0, fs=128.0, order=3):
    nyquist = 0.5 * fs
    normal_cutoff = cutoff / nyquist
    b, a = signal.butter(order, normal_cutoff, btype='low', analog=False)
    y = signal.filtfilt(b, a, series)
    return y